## Retrieval-Augmented Generation (RAG) service

### Vector database setup

In [1]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)

In [2]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [3]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 466662905888375849, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 466662908947070994, 'update_timestamp': 466662908947070994}


In [4]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [5]:
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [6]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

In [7]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

In [8]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [10]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

result

data: [[{'id': 466662905888376184, 'distance': 29.125171661376953, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mog

### Gemini API integration

In [17]:
from dotenv import load_dotenv
load_dotenv()

import os

In [31]:
import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text 
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [32]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonego kontekstu.

Instrukcje:
- Korzystaj tylko z informacji znajdujących się w kontekście.
- Nie używaj wiedzy spoza kontekstu.
- Jeśli kontekst nie zawiera odpowiedzi, napisz:
  "Brak informacji w dostarczonym kontekście."
- Odpowiadaj po polsku.
- Odpowiadaj zwięźle i rzeczowo.

Kontekst:
{context}

Pytanie:
{query}

Odpowiedź:
"""
    return prompt

In [33]:
def rag(model, query: str) -> str:
    # having all prepared functions, you can combine them together and try to build your own RAG!
    result = search(model, query)

    context = "\n\n".join(
        hit["entity"]["text"]
        for hit in result[0]
    )

    prompt = build_prompt(context, query)

    return generate_response(prompt) or "No response generated."

In [34]:
rag(model, "Czym jest sztuczna inteligencja?")

'Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji, takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się i planowanie. Jest to program lub maszyna symulująca inteligencję naturalną, ludzką, której głównym celem jest stworzenie systemów zdolnych do myślenia i podejmowania decyzji w sposób przypominający ludzki, a czasami przewyższający funkcje naturalne.'

In [35]:
rag(model, "Jakie są rodzaje sztucznej inteligencji?")

'Rodzaje sztucznej inteligencji to:\n*   Sztuczna inteligencja słaba (wąska)\n*   Sztuczna inteligencja silna\n*   Sztuczna inteligencja umysłowa\n*   Sztuczna inteligencja zwiększająca (Augmented Intelligence)\n*   Sztuczna inteligencja bezpieczna (Safe AI)\n*   Sztuczna inteligencja odwrócona (Inverse AI)\n*   Sztuczna inteligencja reprezentacyjna (Representational AI)\n*   Sztuczna inteligencja predykcyjna (Predictive AI)'

In [38]:
rag(model, "Jak powstaje sztuczna inteligencja?")

'Sztuczna inteligencja powstaje poprzez tworzenie programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji. Prawdziwy przełom w tej dziedzinie nastąpił dzięki postępowi w algorytmach uczenia maszynowego.'

In [ ]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonego kontekstu.

Instrukcje:
- Korzystaj tylko z informacji znajdujących się w kontekście.
- Nie używaj wiedzy spoza kontekstu.
- Jeśli kontekst nie zawiera odpowiedzi, napisz:
  "Brak informacji w dostarczonym kontekście."
- Odpowiadaj po japońsku.
- Odpowiadaj zwięźle i rzeczowo.

Kontekst:
{context}

Pytanie:
{query}

Odpowiedź:
"""
    return prompt

In [40]:
rag(model, "Czym jest sztuczna inteligencja?")

'人工知能（AI）は、自然な人間の知能をシミュレートする何か（プログラムや機械）であり、人間の知能を必要とするタスクを実行できるコンピュータプログラムの作成に焦点を当てた情報科学の分野です。'

In [43]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonego kontekstu.

Instrukcje:
- Korzystaj tylko z informacji znajdujących się w kontekście.
- Nie używaj wiedzy spoza kontekstu.
- Jeśli kontekst nie zawiera odpowiedzi, napisz:
  "Brak informacji w dostarczonym kontekście."
- Odpowiadaj po polsku.
- Odpowiadaj zwięźle i rzeczowo.
- Odpowiadaj w stylu Yody z Gwiezdnych Wojen.

Kontekst:
{context}

Pytanie:
{query}

Odpowiedź:
"""
    return prompt

In [44]:
rag(model, "Czym jest sztuczna inteligencja?")

'Obszar informatyki ona jest, co na tworzeniu programów komputerowych lub maszyn zdolnych do symulowania ludzkiej inteligencji skupia się. Zadania, co ludzkiej inteligencji wymagają, wykonywać ma za cel. Myśleć i decyzje podejmować na sposób ludzki zdolne systemy stworzyć, to główny cel jej jest. Funkcje umysłu niektóre realizować potrafi, nawet naturalne przewyższać.'

In [45]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
Jesteś asystentem odpowiadającym na pytania wyłącznie na podstawie dostarczonego kontekstu.

Instrukcje:
- Korzystaj tylko z informacji znajdujących się w kontekście.
- Nie używaj wiedzy spoza kontekstu.
- Jeśli kontekst nie zawiera odpowiedzi, napisz:
  "Brak informacji w dostarczonym kontekście."
- Odpowiadaj po polsku.
- Odpowiadaj zwięźle i rzeczowo.

Kontekst:
{context}

Pytanie:
{query}

Odpowiedź:
"""
    return prompt

In [46]:
rag(model, "Kto wygrał Wimbledon w 2025 roku?")

'Brak informacji w dostarczonym kontekście.'